In [1]:
!pip install nltk spacy wordcloud textblob scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
pip install spacy

In [2]:
import nltk
import spacy

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("vader_lexicon")

!python -m spacy download en_core_web_sm

ModuleNotFoundError: No module named 'spacy'

In [ ]:
import re

import nltk
import spacy

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import WordCloud

import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob

nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

 # Exploring  Text Data

In [ ]:
print(df["Student Feedback"].head(10))

# Text Cleaning

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+"," ",text)

    text = re.sub(r"[^a-zA-Z ]"," ",text)

    text = re.sub(r"\s+"," ",text)

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [ ]:
df["Clean Feedback"] = df["Student Feedback"].apply(clean_text)

df[["Student Feedback","Clean Feedback"]].head()

# Feedback Length Analysis

In [ ]:
df["Character Count"] = df["Student Feedback"].str.len()

df["Word Count"] = (
    df["Student Feedback"]
      .str.split()
      .str.len()
)

# Most Frequent Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    max_features=20
)

X = vectorizer.fit_transform(df["Clean Feedback"])

word_counts = (
    pd.DataFrame(
        X.toarray(),
        columns=vectorizer.get_feature_names_out()
    )
)

top_words = word_counts.sum().sort_values(ascending=False)

top_words

In [ ]:
plt.figure(figsize=(10,6))

top_words.plot(kind="bar")

plt.title("Most Frequent Words")

plt.show()

# Word Cloud

In [ ]:
text = " ".join(df["Clean Feedback"])

wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    colormap="viridis"
).generate(text)

plt.figure(figsize=(16,8))

plt.imshow(wordcloud)

plt.axis("off")

plt.title("Word Cloud of Student Feedback")

plt.show()

# Sentiment Analysis (TextBlob)

In [ ]:
def sentiment(text):

    polarity = TextBlob(text).sentiment.polarity

    if polarity > 0.2:
        return "Positive"

    elif polarity < -0.2:
        return "Negative"

    return "Neutral"

In [ ]:
df["Sentiment"] = (
    df["Student Feedback"]
      .apply(sentiment)
)

df["Sentiment"].value_counts()

In [ ]:
plt.figure(figsize=(6,6))

df["Sentiment"].value_counts().plot(
    kind="pie",
    autopct="%1.1f%%"
)

plt.ylabel("")

plt.title("Sentiment Distribution")

plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=500
)

tfidf_matrix = tfidf.fit_transform(df["Clean Feedback"])

print(tfidf_matrix.shape)

# Keyword Extraction

In [ ]:
feature_names = tfidf.get_feature_names_out()

importance = (
    tfidf_matrix.sum(axis=0)
      .A1
)

keywords = pd.DataFrame({

    "Keyword":feature_names,

    "Importance":importance

})

keywords = keywords.sort_values(
    "Importance",
    ascending=False
)

keywords.head(20)

In [ ]:
sample = df["Student Feedback"].iloc[0]

doc = nlp(sample)

for ent in doc.ents:

    print(ent.text, ent.label_)

In [ ]:
df.to_csv(
    "student_feedback_nlp.csv",
    index=False
)